# Chapter 4: Entropy and Purity - Code Examples

Measuring "how quantum" a state is:
- **Purity**: $\gamma = \text{Tr}(\rho^2)$ — ranges from $1/d$ to $1$
- **Entropy**: $S = -\text{Tr}(\rho \log \rho)$ — ranges from $0$ to $\log_2 d$

In [1]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

## Core Functions

In [2]:
def von_neumann_entropy(rho):
    """Von Neumann entropy in bits: S = -Tr(ρ log₂ ρ)"""
    eigenvalues = np.linalg.eigvalsh(rho)
    eigenvalues = eigenvalues[eigenvalues > 1e-10]
    return -np.sum(eigenvalues * np.log2(eigenvalues))


def shannon_entropy(p):
    """Shannon entropy in bits: H = -Σ p_i log₂ p_i"""
    p = np.array(p)
    p = p[p > 1e-10]
    return -np.sum(p * np.log2(p))


def purity(rho):
    """Purity: γ = Tr(ρ²)"""
    return np.real(np.trace(rho @ rho))


def linear_entropy(rho):
    """Linear entropy: 1 - Tr(ρ²), approximation for near-pure states."""
    return 1 - purity(rho)

## Purity: From Pure to Maximally Mixed

In [3]:
# Pure state
rho_pure = np.array([[1, 0], [0, 0]])

# Partially mixed
rho_partial = np.array([[0.7, 0], [0, 0.3]])

# Maximally mixed (d=2)
rho_max_mixed = np.array([[0.5, 0], [0, 0.5]])

print("Purity comparison (qubit, d=2):")
print("=" * 40)
print(f"{'State':<20} {'Purity':<10} {'Min=1/d':<10}")
print("-" * 40)
print(f"{'Pure |0⟩':<20} {purity(rho_pure):<10.3f} {1 / 2:<10.3f}")
print(f"{'Partial (70/30)':<20} {purity(rho_partial):<10.3f}")
print(f"{'Max mixed (I/2)':<20} {purity(rho_max_mixed):<10.3f}")
print()
print(f"Note: For d=2, purity ranges from 1/2 (max mixed) to 1 (pure).")

Purity comparison (qubit, d=2):
State                Purity     Min=1/d   
----------------------------------------
Pure |0⟩             1.000      0.500     
Partial (70/30)      0.580     
Max mixed (I/2)      0.500     

Note: For d=2, purity ranges from 1/2 (max mixed) to 1 (pure).


## Von Neumann Entropy: The Eigenvalue Story

In [4]:
print("Von Neumann entropy is Shannon entropy of eigenvalues:")
print("=" * 55)

states = [
    ("Pure |0⟩", rho_pure),
    ("Partial (70/30)", rho_partial),
    ("Max mixed (I/2)", rho_max_mixed),
]

for name, rho in states:
    eigenvalues = np.linalg.eigvalsh(rho)
    S = von_neumann_entropy(rho)
    print(f"\n{name}:")
    print(f"  Eigenvalues: {eigenvalues}")
    print(f"  Entropy S = -Σ λᵢ log₂(λᵢ) = {S:.3f} bits")

Von Neumann entropy is Shannon entropy of eigenvalues:

Pure |0⟩:
  Eigenvalues: [0. 1.]
  Entropy S = -Σ λᵢ log₂(λᵢ) = -0.000 bits

Partial (70/30):
  Eigenvalues: [0.3 0.7]
  Entropy S = -Σ λᵢ log₂(λᵢ) = 0.881 bits

Max mixed (I/2):
  Eigenvalues: [0.5 0.5]
  Entropy S = -Σ λᵢ log₂(λᵢ) = 1.000 bits


## Diagonal = Classical

For diagonal matrices, von Neumann entropy equals Shannon entropy.

In [5]:
# Classical probability distribution
probs = [0.7, 0.2, 0.1]

# As a diagonal density matrix
rho_diag = np.diag(probs)

H_shannon = shannon_entropy(probs)
S_vonneumann = von_neumann_entropy(rho_diag)

print("Diagonal matrix = classical distribution:")
print(f"\nProbabilities: {probs}")
print(f"\nDensity matrix:\n{rho_diag}")
print(f"\nShannon entropy H(p):     {H_shannon:.4f} bits")
print(f"Von Neumann entropy S(ρ): {S_vonneumann:.4f} bits")
print(f"\nThey're equal! (difference: {abs(H_shannon - S_vonneumann):.2e})")

Diagonal matrix = classical distribution:

Probabilities: [0.7, 0.2, 0.1]

Density matrix:
[[0.7 0.  0. ]
 [0.  0.2 0. ]
 [0.  0.  0.1]]

Shannon entropy H(p):     1.1568 bits
Von Neumann entropy S(ρ): 1.1568 bits

They're equal! (difference: 0.00e+00)


## Coherence Reduces Entropy

Adding off-diagonals changes eigenvalues → changes entropy.

In [6]:
# Same diagonal, different off-diagonals
rho_no_coherence = np.array([[0.5, 0.0], [0.0, 0.5]])

rho_partial_coherence = np.array([[0.5, 0.25], [0.25, 0.5]])

rho_max_coherence = np.array([[0.5, 0.5], [0.5, 0.5]])

print("How coherence affects entropy (same diagonal):")
print("=" * 60)

for name, rho in [
    ("No coherence", rho_no_coherence),
    ("Partial coherence", rho_partial_coherence),
    ("Max coherence", rho_max_coherence),
]:
    eigs = np.linalg.eigvalsh(rho)
    S = von_neumann_entropy(rho)
    gamma = purity(rho)
    print(f"\n{name}:")
    print(f"  ρ = {rho[0]}")
    print(f"      {rho[1]}")
    print(f"  Eigenvalues: {eigs}")
    print(f"  Entropy: {S:.3f} bits, Purity: {gamma:.3f}")

How coherence affects entropy (same diagonal):

No coherence:
  ρ = [0.5 0. ]
      [0.  0.5]
  Eigenvalues: [0.5 0.5]
  Entropy: 1.000 bits, Purity: 0.500

Partial coherence:
  ρ = [0.5  0.25]
      [0.25 0.5 ]
  Eigenvalues: [0.25 0.75]
  Entropy: 0.811 bits, Purity: 0.625

Max coherence:
  ρ = [0.5 0.5]
      [0.5 0.5]
  Eigenvalues: [0. 1.]
  Entropy: -0.000 bits, Purity: 1.000


## Signal State Overlap → Quantum Advantage

The key connection to computational mechanics.

In [7]:
print("Signal state overlap vs quantum complexity")
print("=" * 55)
print(f"{'θ (deg)':<10} {'Overlap':<12} {'C_q (bits)':<12} {'C_μ (bits)':<12} {'Advantage':<10}")
print("-" * 55)

for theta_deg in [0, 15, 30, 45]:
    theta = np.radians(theta_deg)

    # Signal states parameterized by angle
    s0 = np.array([np.cos(theta), np.sin(theta)])
    s1 = np.array([np.sin(theta), np.cos(theta)])

    overlap = np.abs(np.dot(s0, s1))

    # Quantum density matrix: ρ = 0.5|s₀⟩⟨s₀| + 0.5|s₁⟩⟨s₁|
    rho = 0.5 * np.outer(s0, s0) + 0.5 * np.outer(s1, s1)

    C_q = von_neumann_entropy(rho)
    C_mu = 1.0  # H([0.5, 0.5]) = 1 bit

    print(f"{theta_deg:<10} {overlap:<12.3f} {C_q:<12.3f} {C_mu:<12.3f} {C_mu - C_q:<10.3f}")

Signal state overlap vs quantum complexity
θ (deg)    Overlap      C_q (bits)   C_μ (bits)   Advantage 
-------------------------------------------------------
0          0.000        1.000        1.000        0.000     
15         0.500        0.811        1.000        0.189     
30         0.866        0.355        1.000        0.645     
45         1.000        -0.000       1.000        1.000     


In [8]:
# Visualize the relationship
thetas = np.linspace(0, np.pi / 4, 50)
overlaps = []
C_qs = []

for theta in thetas:
    s0 = np.array([np.cos(theta), np.sin(theta)])
    s1 = np.array([np.sin(theta), np.cos(theta)])

    overlaps.append(np.abs(np.dot(s0, s1)))

    rho = 0.5 * np.outer(s0, s0) + 0.5 * np.outer(s1, s1)
    C_qs.append(von_neumann_entropy(rho))

print("\nAs overlap increases from 0 to 1:")
print(f"  C_q drops from {C_qs[0]:.3f} to {C_qs[-1]:.3f} bits")
print(f"  Quantum advantage grows from {1 - C_qs[0]:.3f} to {1 - C_qs[-1]:.3f} bits")
print("\nMore overlap = more compression = bigger quantum advantage!")


As overlap increases from 0 to 1:
  C_q drops from 1.000 to -0.000 bits
  Quantum advantage grows from 0.000 to 1.000 bits

More overlap = more compression = bigger quantum advantage!


## Key Takeaway

> **Purity and entropy quantify the classical/quantum boundary.**
>
> - Diagonal matrices: von Neumann = Shannon (classical)
> - Off-diagonals change eigenvalues → change entropy
> - More coherence → lower entropy → more "quantum"
>
> In computational mechanics:
> - Signal state overlap creates off-diagonals
> - Off-diagonals compress eigenvalues
> - Compressed eigenvalues → $C_q < C_\mu$